# Introduction (updated from original)

This notebook takes the single species network produced in NetColoc and generates a hierarchical systems map of protein communities. The systems map is annotated using GO enrichment to identify conserved functions underlying compulsion/psychiatric disorders in humans and dogs. We validate the role of protein communities by examining the phenotypic response in mice upon genetic perturbation.

**Additional Visualizations**  
* All visualizations of the network, hierarchy and communities can be found in the accompanying Cytoscape file.

# Set Up

In [ ]:
#to get it to work again: DO THIS ONCE?
#step 1
source /proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/netcoloc_env/bin/activate

#step 2
pip install --force-reinstall jupyter

#step 3
pip show jupyter

#step 4
python -m ipykernel install --user --name=netcoloc_env --display-name "netcoloc_env (Python 3.6.8)"


In [ ]:
##
ml bioinfo-tools gcc/8.4.0

In [ ]:
# load required libraries
import os
import sys
import ndex2
from getpass import getpass
import cdapsutil
import pandas as pd
import mygene
mg = mygene.MyGeneInfo()
from gprofiler import GProfiler
gp = GProfiler("MyToolName/0.1")

import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import fdrcorrection

In [2]:
# Set up directory paths. Defaults to the Data/ and Figures/ folders within the repository. 
# #i.e. /proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/CrossSpeciesBMI
cwd = os.path.dirname(os.getcwd())
DATADIR = os.path.join(cwd, "Data/")
FIGDIR = os.path.join(cwd, "Figures/")

In [3]:
# Import functions specific to this study
sys.path.append(cwd)
from updated_netcoloc_functions import *
from analysis_functions import *
from plotting_functions import *

## Load input gene sets and interaction network

In [4]:
#KT###  upload my NPS-scores:
# with the header: gene, NPS_r, NPS_h, NPS_hr (calling dog a rat)
nps_df = pd.read_csv(DATADIR + "outputs/ocd_ccd_zcomb_z12_251022.txt", sep = "\t", index_col=0)
nps_df.head(5)

,NPS_r,NPS_h,NPS_hr
gene,,,
TGFBR1,2.074439,0.286812,0.594974
TGFBR2,1.872690,-0.609285,-1.141002
SMAD3,1.040744,0.070344,0.073211
TGFB1,-0.019932,-0.318449,0.006347
SMAD2,1.073654,-0.977044,-1.049007


In [5]:
## Load the final seed gene lists

#I also need the info from seed genes:
# load human seed genes
seed_bin_human_OCD = pd.read_csv('/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/NetColoc/data/OCDgenes_v4.txt')
print("Number of human OCD seeds:", len(seed_bin_human_OCD))

#remove genes not found in PCNet:
seed_bin_human_OCD = seed_bin_human_OCD[seed_bin_human_OCD['gene'].isin(nps_df.index)]
print("Number of human OCD seeds:", len(seed_bin_human_OCD))


Number of human OCD seeds: 48
Number of human OCD seeds: 48


In [6]:
# If you receive a connection error, please retry.
pc_nodes, G_PC = load_pcnet()

number of nodes:
19267

number of edges:
3852119


## Load the OCD only network

Note: the conserved network will remain private until publication. As an alternative to importing the network from NDEx it can be loaded from a stored file below (Note: this network does not include nodes that are not part of the largest connected component). 

In [7]:
# KT run this: #OBS the uuid is now for our OCD-only:# import the conserved network updated 2025-11-?
#ndex_server='public.ndexbio.org'
#ndex_user=getpass("Username:")
#ndex_password=getpass("Password:")
#G_BMI_coloc_cx = ndex2.create_nice_cx_from_server(
#            ndex_server, 
#            username=ndex_user, 
#            password=ndex_password, 
#            uuid='d539acd3-bbca-11f0-a218-005056ae3c32'
#        )

# print out interactome num nodes and edges for diagnostic purposes
#G_BMI_coloc = G_BMI_coloc_cx.to_networkx()
#print('number of nodes:', len(G_BMI_coloc.nodes))
#print('\nnumber of edges:', len(G_BMI_coloc.edges))

number of nodes: 60

number of edges: 67


# Hierarchical Community Detection with HiDeF

**Note:** Load output from NetColoc 

In [8]:
G_BMI_hier = pd.read_csv('/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/NetColoc/example_notebooks/OCD_ONLY_Network_hierarchy_data.tsv', sep ="\t", index_col=0)
G_BMI_hier.head()

,CD_MemberList,CD_MemberList_Size,CD_Labeled,CD_MemberList_LogSize,CD_CommunityName,CD_AnnotatedMembers,CD_AnnotatedMembers_Size,CD_AnnotatedMembers_Overlap,CD_AnnotatedMembers_Pvalue,HiDeF_persistence,represents,name
0,ZNF804B DNAJA2 ING5 EED ATP1A2 TMEM80 ACACA RG...,361,False,8.496,NaN,NaN,0,0.0,0.0,0,C361,C361
1,NCKIPSD ZNF804B DNAJA2 ITPKC ING5 H3-3B TMEM10...,257,False,8.006,NaN,NaN,0,0.0,0.0,137,C362,C362
2,THAP9 PARM1 ZNF804B AGAP7P LRFN2 APBB3 PGBD4 L...,63,False,5.977,NaN,NaN,0,0.0,0.0,8,C363,C363
3,ZNF232 ZKSCAN4 ZSCAN16 C5orf63 ZNF804A LRRC61 ...,61,False,5.931,NaN,NaN,0,0.0,0.0,101,C364,C364
4,THAP9 ZNF804A PGBD4 SPINDOC TMEM161B ZSCAN31 G...,24,False,4.585,NaN,NaN,0,0.0,0.0,9,C372,C372


In [9]:
## Load systems map from netcoloc run:
G_BMI_hier_df = pd.read_csv("/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/CrossSpeciesBMI/Data/output/OCD_ONLY_systems_map.txt", sep="\t", index_col="name")
G_BMI_hier_df.head(5)

,CD_MemberList,CD_MemberList_Size,CD_Labeled,CD_MemberList_LogSize,CD_CommunityName,CD_AnnotatedMembers,CD_AnnotatedMembers_Size,CD_AnnotatedMembers_Overlap,CD_AnnotatedMembers_Pvalue,HiDeF_persistence,represents,name.1,system_ID,num_d1_seeds,frac_d1_seeds,frac_no_seeds
name,,,,,,,,,,,,,,,,
C361,ZNF804B DNAJA2 ING5 EED ATP1A2 TMEM80 ACACA RG...,361,False,8.496,NaN,NaN,0,0.0,0.0,0,C361,C361,0,48,0.132964,0.867036
C362,NCKIPSD ZNF804B DNAJA2 ITPKC ING5 H3-3B TMEM10...,257,False,8.006,NaN,NaN,0,0.0,0.0,137,C362,C362,1,41,0.159533,0.840467
C363,THAP9 PARM1 ZNF804B AGAP7P LRFN2 APBB3 PGBD4 L...,63,False,5.977,NaN,NaN,0,0.0,0.0,8,C363,C363,2,5,0.079365,0.920635
C364,ZNF232 ZKSCAN4 ZSCAN16 C5orf63 ZNF804A LRRC61 ...,61,False,5.931,NaN,NaN,0,0.0,0.0,101,C364,C364,3,4,0.065574,0.934426
C372,THAP9 ZNF804A PGBD4 SPINDOC TMEM161B ZSCAN31 G...,24,False,4.585,NaN,NaN,0,0.0,0.0,9,C372,C372,4,2,0.083333,0.916667


## Seed gene fractions

In [10]:
hier_df_genes = G_BMI_hier_df.loc[:, [ "frac_d1_seeds"]]
hier_df_genes.head()

,frac_d1_seeds
name,
C361,0.132964
C362,0.159533
C363,0.079365
C364,0.065574
C372,0.083333


# Community Annotation

## Initial Annotation with GO

Annotate with moderately sized systems (between 50 to 1000 genes per system) if the community shows a significant ($p<1\times 10^{-4}$) enrichment for the term. Also require that the GO term and community and share at least 3 genes to increase the stringency of the annotation. Label the system using the GO term that meets these criteria, and has the highest sum of precision and recall. Systems which have no GO terms meeting these criteria are labeled with their unique system ID.

**Note:**  
Rerunning the annotation may generate slightly different results due to changes in the GO:BP database over time. The original results were generated January 18, 2022.

In [11]:
def filter_go_annotations(go_df, term_min=50, term_max=1000, p_th=1e-4, min_intersection=3):
    """Filters available annotations for a community based on specificity and significance.
    Args:
        go_df (pandas.DataFrame): All available significant GO terms for each community
        term_min (int, optional):   The minimum size of a term to keep. Defaults to 50.
        term_max (int, optional): The maximum size of a term to keep. Defaults to 1000.
        p_th (float, optional): The significance threshold. Defaults to 1e-4.
        min_intersection (int, optional): Minimum number of community terms annotated to the GO term. Defaults to 3.

    Returns:
        pandas.DataFrame: A filter dataframe of GO annotations per community, sorted by sum of precision and recall.
    """
    go_df = go_df[(go_df['term_size'] <= term_max) & (go_df['term_size'] >= term_min)]
    go_df = go_df[go_df['intersection_size'] >= min_intersection]
    go_df = go_df[go_df['p_value'] < p_th] # set a stringent pvalue threshold
    go_df['sum_PR'] = go_df['recall'] + go_df['precision']
    go_df = go_df.sort_values('sum_PR',ascending=False)
    return go_df

In [12]:
G_BMI_hier_df.head()

,CD_MemberList,CD_MemberList_Size,CD_Labeled,CD_MemberList_LogSize,CD_CommunityName,CD_AnnotatedMembers,CD_AnnotatedMembers_Size,CD_AnnotatedMembers_Overlap,CD_AnnotatedMembers_Pvalue,HiDeF_persistence,represents,name.1,system_ID,num_d1_seeds,frac_d1_seeds,frac_no_seeds
name,,,,,,,,,,,,,,,,
C361,ZNF804B DNAJA2 ING5 EED ATP1A2 TMEM80 ACACA RG...,361,False,8.496,NaN,NaN,0,0.0,0.0,0,C361,C361,0,48,0.132964,0.867036
C362,NCKIPSD ZNF804B DNAJA2 ITPKC ING5 H3-3B TMEM10...,257,False,8.006,NaN,NaN,0,0.0,0.0,137,C362,C362,1,41,0.159533,0.840467
C363,THAP9 PARM1 ZNF804B AGAP7P LRFN2 APBB3 PGBD4 L...,63,False,5.977,NaN,NaN,0,0.0,0.0,8,C363,C363,2,5,0.079365,0.920635
C364,ZNF232 ZKSCAN4 ZSCAN16 C5orf63 ZNF804A LRRC61 ...,61,False,5.931,NaN,NaN,0,0.0,0.0,101,C364,C364,3,4,0.065574,0.934426
C372,THAP9 ZNF804A PGBD4 SPINDOC TMEM161B ZSCAN31 G...,24,False,4.585,NaN,NaN,0,0.0,0.0,9,C372,C372,4,2,0.083333,0.916667


In [13]:
# Annotate with GO terms gprofiler annotation of clusters
system_names = {}
all_results = []
BMI_hier_df = G_BMI_hier_df
for comm in BMI_hier_df.index.tolist():
    # get the genes in the community
    if isinstance(BMI_hier_df['CD_MemberList'].loc[comm], str):
        focal_genes=BMI_hier_df['CD_MemberList'].loc[comm].split(' ')
    else:
        focal_genes=BMI_hier_df['CD_MemberList'].loc[comm]
    print(comm,":", len(focal_genes))
    system_names[comm] = comm # default is to keep identifier as name, will be replaced if confident annotation available
    if len(focal_genes)>2:
        # get all available GO:BP annotations
        gp_temp = pd.DataFrame(gp.profile(focal_genes,significance_threshold_method='fdr',
                                            sources=['GO:BP'], no_evidences=False))
        if len(gp_temp)>0: # make sure data is not empty
            # filter and append the results
            gp_temp["name.1"] = comm
            all_results.append(gp_temp)
            gp_temp = filter_go_annotations(gp_temp, 50, 1000, 1e-3, 3)
            if len(gp_temp)>1:
                system_names[comm] = gp_temp.head(1)['name'].tolist()[0]
                #all_results.append(gp_temp)
go_results = pd.concat(all_results)
go_results.to_csv(DATADIR+"outputs/OCD_ONLY_hierachy_full_GO_enrichment.tsv", index=True, sep="\t") #updated 2025-11-04

C361 : 361


C362 : 257
C363 : 63
C364 : 61
C372 : 24
C404 : 6
C377 : 15
C412 : 5
C389 : 9
C392 : 9
C393 : 9
C402 : 9
C409 : 6
C413 : 5
C414 : 5
C421 : 4
C431 : 4
C437 : 4
C441 : 4
C366 : 35
C367 : 34
C369 : 30
C373 : 23
C374 : 19
C376 : 18
C381 : 12
C365 : 38
C383 : 12
C368 : 31
C388 : 10
C417 : 5
C380 : 14
C370 : 28
C390 : 9
C391 : 9
C405 : 6
C385 : 11
C371 : 24
C375 : 19
C378 : 14
C379 : 14
C397 : 8
C395 : 8
C427 : 4
C382 : 12
C384 : 12
C415 : 5
C386 : 11
C411 : 5
C403 : 7
C396 : 8
C401 : 7
C408 : 6
C429 : 4
C435 : 4
C438 : 4
C424 : 4
C400 : 7
C420 : 5
C440 : 4


In [13]:
#read the one run instead of rerunning:
go_results = pd.read_csv(DATADIR+"outputs/OCD_ONLY_hierachy_full_GO_enrichment.tsv",  sep="\t", index_col=0)
go_results.head(5)

,description,effective_domain_size,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,evidences,name.1
0,"""Cell-cell signaling in either direction acros...",21026,173,"['GRIN2B', 'FGF12', 'CADPS', 'GRM4', 'GRIA3', ...",trans-synaptic signaling,GO:0099537,1.026574e-132,['GO:0099536'],0.404206,query_1,428,0.221511,True,GO:BP,781,"[['ISS', 'ISO', 'IBA', 'TAS', 'NAS', 'IEA'], [...",C462
1,"""Cell-cell signaling to, from or within a syna...",21026,175,"['GRIN2B', 'FGF12', 'CADPS', 'GRM4', 'GRIA3', ...",synaptic signaling,GO:0099536,1.026574e-132,['GO:0007267'],0.408879,query_1,428,0.216853,True,GO:BP,807,"[['ISS', 'ISO', 'IBA', 'TAS', 'NAS', 'IEA'], [...",C462
2,"""Cell-cell signaling from pre to post-synapse,...",21026,172,"['GRIN2B', 'FGF12', 'CADPS', 'GRM4', 'GRIA3', ...",anterograde trans-synaptic signaling,GO:0098916,2.747700e-132,['GO:0099537'],0.401869,query_1,428,0.221935,True,GO:BP,775,"[['ISS', 'ISO', 'IBA', 'TAS', 'NAS', 'IEA'], [...",C462
3,"""The vesicular release of classical neurotrans...",21026,172,"['GRIN2B', 'FGF12', 'CADPS', 'GRM4', 'GRIA3', ...",chemical synaptic transmission,GO:0007268,2.747700e-132,['GO:0098916'],0.401869,query_1,428,0.221935,True,GO:BP,775,"[['ISS', 'ISO', 'IBA', 'TAS', 'NAS', 'IEA'], [...",C462
4,"""Any process that mediates the transfer of inf...",21026,190,"['GRIN2B', 'FGF12', 'CADPS', 'GRM4', 'GRIA3', ...",cell-cell signaling,GO:0007267,3.490877e-110,"['GO:0007154', 'GO:0023052']",0.443925,query_1,428,0.141580,True,GO:BP,1342,"[['ISS', 'ISO', 'IBA', 'TAS', 'NAS', 'IEA'], [...",C462


# Validation


## Load Mouse Genome Database (MGD) data

**Note:** we use a stable version of MGD downloaded on April 24, 2022. If using an updated dataset, slight variations in results are expected due to recent data submission to MGD. 

In [14]:
# load mappings between genes and mammalian phenotypes
mgi_df = load_MGI_mouseKO_data(url='http://www.informatics.jax.org/downloads/reports/MGI_PhenoGenoMP.rpt', 
                                map_using="mgi", update=False, data_loc=DATADIR+"Reference/") #changed update to True when I have downloaded it once at Jan 30th 2025
print("Number of gene-phenotype associations:", len(mgi_df))

Number of gene-phenotype associations: 399047


**Node Updating**  
The gene symbols listed under human orthologs in the MGD are more up to date than the gene symbols in PCNet. Therefore, to ensure correct mapping of our conserved network to MGD we map the human ortholog column to the symbols in PCNet using MyGeneInfo. Some gene symbols return duplicate entries via MyGeneInfo which we resolved manually (see `symbol_list` below).

In [15]:
symbol_list = {'GARS': 'GARS1', 'GGT2': 'GGT2P','MARS': 'MARS1', 'HIST1H2BC': 'H2BC4', 'GBA': 'GBA1', 'HIST1H2BG': 'H2BC8',
'HIST1H2BF': 'H2BC7', 'HIST1H2BE': 'H2BC6', 'HIST1H2BI': 'H2BC10', 'CBWD3': 'ZNG1C', 'BHLHB9': 'GPRASP3', 'ELOA3': 'ELOA3P',
'ELOA3B': 'ELOA3BP', 'SMIM11B': 'SMIM11', 'CBSL': 'CBS', 'C2orf27B': 'CDRT15P3', 'LINC00846': 'TCP10L'}
updated_pc_nodes, _ = update_nodes(pc_nodes, symbol_list)

167 input query terms found dup hits:	[('CAST', 2), ('DUX4L9', 4), ('IGHV1-69', 2), ('IGHV3-48', 2), ('IGHV3-33', 2), ('IGHV3-30', 2), ('I
87 input query terms found no hit:	['MFSD10', 'ILVBL', 'ND5', 'COX2', 'NDUFA4', 'BMT2', 'NEPRO', 'ARMT1', 'NDUFA4L2', 'TMEM199', 'C15or
5 input query terms found dup hits:	[('COX2', 2), ('NDUFA4', 2), ('C18orf21', 2), ('C17orf49', 2), ('CIR1', 2)]
10 input query terms found no hit:	['ND5', 'ND2', 'ND4', 'ND3', 'ND4L', 'ND6', 'CYTB', 'ATP8', 'ATP6', 'TEC_100124696']


In [16]:
# updated the genotype-phenotype identifiers
mgi_df = change_symbols(mgi_df, updated_pc_nodes)

In [17]:
# load the mammalian phenotype ontology
MPO = load_MPO(url='http://www.informatics.jax.org/downloads/reports/MPheno_OBO.ontology', 
                        use_genes=True, mapping=mgi_df, data_loc=DATADIR+"Reference/", update=False) #changed update to True when I have downloaded it once at Jan 30th 2025

15418


In [18]:
# summarize the total evidence per gene and term
term_counts, gene_mapping, term_mapping = genes_per_node(MPO)

In [19]:
# construct a graph object of the MPO
mp_graph = get_mp_graph(DATADIR+"Reference/parsed_mp.txt")

## Perform enrichment of Mammalian Phenotypes

### All phenotypes in MGD:

In [20]:
### Obs this is already done for all phenotypes, read in the results below:
bmi_phenotypes = []
########## for each community, calculate the enrichment for all phenotypes##############
for community in tqdm(G_BMI_hier_df.index):
    x = community_term_enrichment(community,  G_BMI_hier_df, MPO, mgi_df, term_counts, gene_mapping, G_int=G_PC)
    x = x.assign(name=community)
    bmi_phenotypes.append(x)
bmi_pheno_results = pd.concat(bmi_phenotypes)
bmi_pheno_results.to_csv(DATADIR + "outputs/OCD_ONLY_hierarchy_full_MGD_enrichment_results.tsv", sep="\t") 

#bmi_pheno_results = pd.read_csv(DATADIR+"outputs/OCD_ONLY_hierarchy_full_MGD_enrichment_results.tsv", sep="\t", index_col=0)
bmi_pheno_results.head()
    

  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [01:06<00:00,  1.11s/it]


,observed,total,OR,OR_p,OR_CI_lower,OR_CI_upper,hyper_p,sig_5e6,description,size,name
MP:0000001,223,14058,0.592298,1.743490e-06,0.477854,0.734151,0.999998,False,mammalian phenotype,361,C361
MP:0005378,97,6707,0.683487,1.477877e-03,0.540549,0.864221,0.999311,False,growth/size/body region phenotype,361,C361
MP:0005386,95,5760,0.834762,1.341539e-01,0.659070,1.057291,0.926561,False,behavior/neurological phenotype,361,C361
MP:0004924,95,5758,0.835183,1.352444e-01,0.659402,1.057824,0.925960,False,abnormal behavior,361,C361
MP:0005376,93,7424,0.547906,7.006338e-07,0.431996,0.694916,1.000000,False,homeostasis/metabolism phenotype,361,C361


In [24]:
# assign the community annotations
bmi_pheno_results = bmi_pheno_results.merge(final_annotations, left_on="name", right_index=True)
bmi_pheno_results.value_counts("description")

description
mammalian phenotype                      61
behavior/neurological phenotype          58
abnormal nervous system physiology       58
nervous system phenotype                 58
abnormal behavior                        58
                                         ..
abnormal thymus involution                1
abnormal granulosa cell proliferation     1
decreased circulating ghrelin level       1
abnormal thyroid gland morphology         1
dilated respiratory conducting tube       1
Length: 4275, dtype: int64